In [ ]:
!pip install -q transformers torch torchvision accelerate


In [ ]:
!pip install -q gdown

In [ ]:
!gdown --id 12QAt09cAXPoEgKGCEQm1tYuTokCKBnjb

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=12QAt09cAXPoEgKGCEQm1tYuTokCKBnjb
From (redirected): https://drive.google.com/uc?id=12QAt09cAXPoEgKGCEQm1tYuTokCKBnjb&confirm=t&uuid=616e988c-00a2-40b4-92d4-04e59a39ec27
To: /content/data_object_image_2.zip
  1% 124M/12.6G [00:04<10:23, 20.0MB/s]

In [ ]:
!ls

In [ ]:
!unzip data_object_image_2.zip

Streaming output truncated to the last 5000 lines.
 extracting: testing/image_2/004818.png  
 extracting: testing/image_2/003183.png  
 extracting: testing/image_2/007384.png  
 extracting: testing/image_2/002195.png  
 extracting: testing/image_2/002058.png  
 extracting: testing/image_2/006043.png  
 extracting: testing/image_2/006951.png  
 extracting: testing/image_2/002983.png  
 extracting: testing/image_2/007502.png  
 extracting: testing/image_2/007116.png  
 extracting: testing/image_2/006136.png  
 extracting: testing/image_2/006982.png  
 extracting: testing/image_2/000618.png  
 extracting: testing/image_2/005595.png  
 extracting: testing/image_2/000198.png  
 extracting: testing/image_2/005881.png  
 extracting: testing/image_2/005327.png  
 extracting: testing/image_2/005955.png  
 extracting: testing/image_2/000106.png  
 extracting: testing/image_2/007168.png  
 extracting: testing/image_2/007269.png  
 extracting: testing/image_2/006352.png  
 extracting: testing/imag

In [ ]:
!gdown --id 1GRj7CsQajuaaBynxLQuHkO0KAPEVwCnD

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1GRj7CsQajuaaBynxLQuHkO0KAPEVwCnD
From (redirected): https://drive.google.com/uc?id=1GRj7CsQajuaaBynxLQuHkO0KAPEVwCnD&confirm=t&uuid=82dfe02b-6db0-491c-9638-83839e8b1319
To: /content/kitti_loader.py
100% 4.27k/4.27k [00:00<00:00, 15.8MB/s]


In [ ]:
!ls

data_object_image_2.zip  kitti_loader.py  sample_data  testing	training


In [ ]:
from kitti_loader import KITTILoader

print(KITTILoader)

<class 'kitti_loader.KITTILoader'>


In [ ]:
from transformers import pipeline

from PIL import Image

import numpy as np

import os

import torch

device = 0 if torch.cuda.is_available() else -1

print(f'Используем: {"GPU" if device == 0 else "CPU"}')


Используем: GPU


 ## Загружаем depth-estimation Small-версию (быстрее, помещается на T4)

In [ ]:
pipe = pipeline(

    task="depth-estimation",

    model="depth-anything/Depth-Anything-V2-Small-hf",

    device=device

)

print('✅ Depth Anything v2 загружен')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ Depth Anything v2 загружен


In [ ]:
def predict_depth(image_path, save_path=None):

    """
    Предсказать depth для одного изображения
    Возвращает np.array (H, W) с относительной глубиной 0–1.
    """

    image = Image.open(image_path).convert('RGB')

    result = pipe(image)

    depth = np.array(result['depth'])  # (H, W), относительная



    # Нормализуем в 0–1
    depth_normalized = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)



    if save_path:

        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        np.save(save_path, depth_normalized.astype(np.float32))

    return depth_normalized


In [ ]:
import os

os.makedirs("/content/depth_pred_kitti", exist_ok=True)

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
KITTI_RGB = '/content/training/image_2'

DEPTH_PRED_KITTI = '/content/depth_pred_kitti'

In [ ]:
images = sorted(os.listdir(KITTI_RGB))[:1000]

In [ ]:
print(images)

['000000.png', '000001.png', '000002.png', '000003.png', '000004.png', '000005.png', '000006.png', '000007.png', '000008.png', '000009.png', '000010.png', '000011.png', '000012.png', '000013.png', '000014.png', '000015.png', '000016.png', '000017.png', '000018.png', '000019.png', '000020.png', '000021.png', '000022.png', '000023.png', '000024.png', '000025.png', '000026.png', '000027.png', '000028.png', '000029.png', '000030.png', '000031.png', '000032.png', '000033.png', '000034.png', '000035.png', '000036.png', '000037.png', '000038.png', '000039.png', '000040.png', '000041.png', '000042.png', '000043.png', '000044.png', '000045.png', '000046.png', '000047.png', '000048.png', '000049.png', '000050.png', '000051.png', '000052.png', '000053.png', '000054.png', '000055.png', '000056.png', '000057.png', '000058.png', '000059.png', '000060.png', '000061.png', '000062.png', '000063.png', '000064.png', '000065.png', '000066.png', '000067.png', '000068.png', '000069.png', '000070.png', '0000

In [ ]:
for img_name in tqdm(images):

    rgb_path = f'{KITTI_RGB}/{img_name}'

    save_path = f'{DEPTH_PRED_KITTI}/{img_name.replace(".png", ".npy")}'

    if not os.path.exists(save_path):

        predict_depth(rgb_path, save_path)

print('✅ Инференс на KITTI завершён')

In [ ]:
!zip -r depth_pred_kitti.zip /content/depth_pred_kitti

  adding: content/depth_pred_kitti/ (stored 0%)
  adding: content/depth_pred_kitti/000894.npy (deflated 96%)
  adding: content/depth_pred_kitti/000078.npy (deflated 96%)
  adding: content/depth_pred_kitti/000515.npy (deflated 96%)
  adding: content/depth_pred_kitti/000960.npy (deflated 95%)
  adding: content/depth_pred_kitti/000200.npy (deflated 95%)
  adding: content/depth_pred_kitti/000254.npy (deflated 96%)
  adding: content/depth_pred_kitti/000440.npy (deflated 96%)
  adding: content/depth_pred_kitti/000934.npy (deflated 94%)
  adding: content/depth_pred_kitti/000132.npy (deflated 96%)
  adding: content/depth_pred_kitti/000729.npy (deflated 97%)
  adding: content/depth_pred_kitti/000867.npy (deflated 95%)
  adding: content/depth_pred_kitti/000067.npy (deflated 96%)
  adding: content/depth_pred_kitti/000403.npy (deflated 97%)
  adding: content/depth_pred_kitti/000913.npy (deflated 95%)
  adding: content/depth_pred_kitti/000847.npy (deflated 95%)
  adding: content/depth_pred_kitti/00

In [21]:
import os
import shutil

selected_dir = "/content/kitti_1000_images"
os.makedirs(selected_dir, exist_ok=True)

images = sorted(os.listdir(KITTI_RGB))[:1000]

for img_name in images:
    src = os.path.join(KITTI_RGB, img_name)
    dst = os.path.join(selected_dir, img_name)

    shutil.copy(src, dst)

In [22]:
!zip -r kitti_1000_images.zip /content/kitti_1000_images

  adding: content/kitti_1000_images/ (stored 0%)
  adding: content/kitti_1000_images/000807.png (deflated 4%)
  adding: content/kitti_1000_images/000400.png (deflated 1%)
  adding: content/kitti_1000_images/000787.png (deflated 1%)
  adding: content/kitti_1000_images/000704.png (deflated 2%)
  adding: content/kitti_1000_images/000443.png (deflated 3%)
  adding: content/kitti_1000_images/000710.png (deflated 3%)
  adding: content/kitti_1000_images/000998.png (deflated 2%)
  adding: content/kitti_1000_images/000224.png (deflated 1%)
  adding: content/kitti_1000_images/000247.png (deflated 2%)
  adding: content/kitti_1000_images/000945.png (deflated 3%)
  adding: content/kitti_1000_images/000809.png (deflated 5%)
  adding: content/kitti_1000_images/000616.png (deflated 2%)
  adding: content/kitti_1000_images/000726.png (deflated 0%)
  adding: content/kitti_1000_images/000674.png (deflated 2%)
  adding: content/kitti_1000_images/000961.png (deflated 4%)
  adding: content/kitti_1000_images/